In [1]:
# %% Cell 1 — Imports & file discovery
import pandas as pd
import numpy as np
import h3
import glob
import os

POOL_DIR = "./poolingdata"
parquet_files = sorted(glob.glob(os.path.join(POOL_DIR, "*.parquet")))
print(f"Found {len(parquet_files)} parquet files in {POOL_DIR}")

Found 11 parquet files in ./poolingdata


In [2]:
# %% Cell 2 — Phase 1: Build df2 (unique rides) from ALL files
# We stream file-by-file, extract one row per rid, then concat + deduplicate.
# Memory-efficient: only keep the small df2 parts, discard the big df each loop.

df2_parts = []

for i, fp in enumerate(parquet_files):
    chunk = pd.read_parquet(fp, engine="pyarrow")
    # one row per ride from this chunk
    chunk_unique = chunk.drop_duplicates(subset="rid", keep="first")
    df2_parts.append(chunk_unique)
    if (i + 1) % 10 == 0 or i == len(parquet_files) - 1:
        print(f"  Scanned {i+1}/{len(parquet_files)} files — "
              f"accumulated {sum(len(p) for p in df2_parts)} unique-ride rows so far")
    del chunk  # free memory

# Concat and keep truly unique rides across all files
df2 = pd.concat(df2_parts, ignore_index=True)
del df2_parts
df2 = df2.drop_duplicates(subset="rid", keep="first").reset_index(drop=True)
print(f"\n✓ df2 built: {df2.shape[0]} unique rides, {df2.shape[1]} columns")
df2.info()

  Scanned 10/11 files — accumulated 61739 unique-ride rows so far
  Scanned 11/11 files — accumulated 67828 unique-ride rows so far

✓ df2 built: 67828 unique rides, 14 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67828 entries, 0 to 67827
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   rid              67828 non-null  object        
 1   driver_id        67828 non-null  object        
 2   ts               67828 non-null  datetime64[ns]
 3   lat              67828 non-null  object        
 4   lon              67828 non-null  object        
 5   rideStatus       67828 non-null  object        
 6   rider_id         67828 non-null  object        
 7   status           67828 non-null  object        
 8   trip_start_lat   67828 non-null  float64       
 9   trip_start_lon   67828 non-null  float64       
 10  trip_end_lat     67828 non-null  float64       
 11  trip_end_lon     678

In [3]:
# %% Cell 3 — Phase 1b: Build journey_coords & journey_hexes from ALL files
# We need ALL trajectory points for each ride to build complete journeys.
# Strategy: stream each file, keep only ON_RIDE rows with slim cols,
# accumulate coords + hexes per rid.

from collections import defaultdict

# Accumulators: rid -> list of (ts, lat, lon)  (we keep ts to sort later)
ride_points = defaultdict(list)

slim_cols = ["rid", "ts", "lat", "lon", "rideStatus"]

for i, fp in enumerate(parquet_files):
    chunk = pd.read_parquet(fp, columns=slim_cols, engine="pyarrow")
    chunk = chunk[chunk["rideStatus"] == "ON_RIDE"].copy()
    chunk["lat"] = pd.to_numeric(chunk["lat"], errors="coerce")
    chunk["lon"] = pd.to_numeric(chunk["lon"], errors="coerce")
    chunk = chunk.dropna(subset=["lat", "lon"])
    chunk["ts"] = pd.to_datetime(chunk["ts"])

    for rid, lat, lon, ts in zip(chunk["rid"], chunk["lat"], chunk["lon"], chunk["ts"]):
        ride_points[rid].append((ts, lat, lon))

    if (i + 1) % 10 == 0 or i == len(parquet_files) - 1:
        print(f"  Accumulated trajectory points from {i+1}/{len(parquet_files)} files — "
              f"{len(ride_points)} distinct rides so far")
    del chunk

print(f"\n✓ Trajectory points collected for {len(ride_points)} rides")

  Accumulated trajectory points from 10/11 files — 61700 distinct rides so far
  Accumulated trajectory points from 11/11 files — 67785 distinct rides so far

✓ Trajectory points collected for 67785 rides


In [4]:
# %% Cell 4 — Build journey_coords & journey_hexes, merge into df2

def _unique_ordered(series):
    """Return unique values preserving first-seen order."""
    seen = set()
    result = []
    for v in series:
        if v not in seen:
            seen.add(v)
            result.append(v)
    return result

journey_data = {}  # rid -> {journey_coords: [...], journey_hexes: [...]}

for rid, points in ride_points.items():
    # Sort by timestamp to ensure chronological order
    points.sort(key=lambda x: x[0])
    coords = [(lat, lon) for _, lat, lon in points]
    hexes = [h3.latlng_to_cell(lat, lon, 9) for lat, lon in coords]
    journey_data[rid] = {
        "journey_coords": coords,
        "journey_hexes": _unique_ordered(hexes),
    }

del ride_points  # free memory

# Add to df2
df2["journey_coords"] = df2["rid"].map(lambda r: journey_data.get(r, {}).get("journey_coords"))
df2["journey_hexes"]  = df2["rid"].map(lambda r: journey_data.get(r, {}).get("journey_hexes"))

# Convert df2 types
df2["lat"] = pd.to_numeric(df2["lat"], errors="coerce")
df2["lon"] = pd.to_numeric(df2["lon"], errors="coerce")
df2["ts"]            = pd.to_datetime(df2["ts"])
df2["trip_end_time"] = pd.to_datetime(df2["trip_end_time"])

print(f"✓ df2 shape after merge: {df2.shape}")
print(f"  Columns: {df2.columns.tolist()}")

# Stats
has_journey = df2["journey_coords"].notna()
print(f"  Rides with journey data: {has_journey.sum()} / {len(df2)}")
if has_journey.any():
    lengths = df2.loc[has_journey, "journey_coords"].apply(len)
    print(f"  journey_coords lengths: min={lengths.min()}, max={lengths.max()}, mean={lengths.mean():.1f}")
    hex_lengths = df2.loc[has_journey, "journey_hexes"].apply(len)
    print(f"  journey_hexes  lengths: min={hex_lengths.min()}, max={hex_lengths.max()}, mean={hex_lengths.mean():.1f}")


✓ df2 shape after merge: (67828, 16)
  Columns: ['rid', 'driver_id', 'ts', 'lat', 'lon', 'rideStatus', 'rider_id', 'status', 'trip_start_lat', 'trip_start_lon', 'trip_end_lat', 'trip_end_lon', 'trip_start_time', 'trip_end_time', 'journey_coords', 'journey_hexes']
  Rides with journey data: 67785 / 67828
  journey_coords lengths: min=1, max=7148, mean=344.8
  journey_hexes  lengths: min=1, max=209, mean=21.3


In [5]:
# %% Cell 5 — Build ride_meta_sim from df2 (same as before)

ride_meta_sim = {}
for _, r in df2.iterrows():
    ride_meta_sim[r['rid']] = {
        'trip_end_time':  r['trip_end_time'],
        'trip_end_lat':   r['trip_end_lat'],
        'trip_end_lon':   r['trip_end_lon'],
        'journey_coords': r['journey_coords'],
        'journey_hexes':  r['journey_hexes'],
    }

print(f"✓ ride_meta_sim built: {len(ride_meta_sim)} rides")

✓ ride_meta_sim built: 67828 rides


In [6]:
# %% Cell 6 — Phase 2: Load slim trajectory data for sweep-line
# Read all files but keep ONLY the 5 columns needed.
# Filter to ON_RIDE, add H3 hex, sort globally by ts.

slim_cols = ["rid", "ts", "lat", "lon", "rideStatus"]
df_parts = []

for i, fp in enumerate(parquet_files):
    chunk = pd.read_parquet(fp, columns=slim_cols, engine="pyarrow")
    chunk = chunk[chunk["rideStatus"] == "ON_RIDE"].copy()
    chunk["lat"] = pd.to_numeric(chunk["lat"], errors="coerce")
    chunk["lon"] = pd.to_numeric(chunk["lon"], errors="coerce")
    chunk = chunk.dropna(subset=["lat", "lon"])
    chunk["ts"] = pd.to_datetime(chunk["ts"])
    # Generate H3 hex
    chunk["curr_hex"] = [
        h3.latlng_to_cell(lat, lon, 9)
        for lat, lon in zip(chunk["lat"], chunk["lon"])
    ]
    df_parts.append(chunk)
    if (i + 1) % 10 == 0 or i == len(parquet_files) - 1:
        print(f"  Loaded {i+1}/{len(parquet_files)} files — "
              f"{sum(len(p) for p in df_parts)} ON_RIDE rows so far")
    del chunk

# We can free journey_data now since ride_meta_sim has everything
del journey_data

df = pd.concat(df_parts, ignore_index=True)
del df_parts

df = df.sort_values("ts").reset_index(drop=True)
print(f"\n✓ df (slim, sorted) ready: {df.shape}")
print(f"  Time range: {df['ts'].min()} → {df['ts'].max()}")
df.info()


  Loaded 10/11 files — 21266805 ON_RIDE rows so far
  Loaded 11/11 files — 23371290 ON_RIDE rows so far

✓ df (slim, sorted) ready: (23371290, 6)
  Time range: 2025-08-02 00:01:15.025000 → 2025-08-02 23:59:59.989000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23371290 entries, 0 to 23371289
Data columns (total 6 columns):
 #   Column      Dtype         
---  ------      -----         
 0   rid         object        
 1   ts          datetime64[ns]
 2   lat         float64       
 3   lon         float64       
 4   rideStatus  object        
 5   curr_hex    object        
dtypes: datetime64[ns](1), float64(2), object(3)
memory usage: 1.0+ GB


In [7]:
# %% Cell 7 — Sweep-line algorithm (UNCHANGED from nypool2.4)
# This is identical to your existing algorithm cell.
# The only input it needs is: df (sorted by ts, with curr_hex) and ride_meta_sim.

import heapq
import math
import requests
from collections import defaultdict

def bearing(lat1, lon1, lat2, lon2):
    dlon = math.radians(lon2 - lon1)
    x = math.sin(dlon) * math.cos(math.radians(lat2))
    y = (math.cos(math.radians(lat1)) * math.sin(math.radians(lat2)) -
         math.sin(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.cos(dlon))
    return math.degrees(math.atan2(x, y)) % 360

def direction_compatible(path_a, path_b, threshold=90):
    bear_a = bearing(*path_a[0], *path_a[-1])
    bear_b = bearing(*path_b[0], *path_b[-1])
    diff = abs(bear_a - bear_b)
    diff = min(diff, 360 - diff)
    return diff < threshold

def path_similarity(h3_path_1, h3_path_2, coords_1, coords_2, k=2):
    if not direction_compatible(coords_1, coords_2, threshold=90):
        return 0.0
    
    set_1 = set()
    for h in h3_path_1:
        set_1.update(h3.grid_disk(h, k))
    
    set_2 = set()
    for h in h3_path_2:
        set_2.update(h3.grid_disk(h, k))
    
    cov_1 = sum(1 for h in h3_path_1 if h in set_2) / len(h3_path_1)
    cov_2 = sum(1 for h in h3_path_2 if h in set_1) / len(h3_path_2)
    
    return min(cov_1, cov_2)

def check_osrm_detour(A_curr, B_curr, A_end, B_end, threshold=30.0):
    def get_route_legs(coords):
        coords_str = ";".join([f"{lon},{lat}" for lat, lon in coords])
        url = f"http://127.0.0.1:5000/route/v1/driving/{coords_str}?overview=false"
        try:
            res = requests.get(url).json()
            if res.get("code") == "Ok":
                return [leg["distance"] for leg in res["routes"][0]["legs"]]
        except Exception:
            pass
        return None

    legs1 = get_route_legs([A_curr, B_curr, A_end, B_end])
    legs2 = get_route_legs([A_curr, B_curr, B_end, A_end])
    legsA = get_route_legs([A_curr, A_end])
    
    if not legs1 or not legs2 or not legsA:
        return False
        
    direct_A = legsA[0]
    direct_B = legs2[1]  
    
    if direct_A <= 0 or direct_B <= 0:
        return False
        
    dist1 = sum(legs1)
    dist2 = sum(legs2)
    
    if dist1 < dist2:
        detour_A = ((legs1[0] + legs1[1]) - direct_A) / direct_A * 100
        detour_B = ((legs1[1] + legs1[2]) - direct_B) / direct_B * 100
    else:
        detour_A = ((legs2[0] + legs2[1] + legs2[2]) - direct_A) / direct_A * 100
        detour_B = 0.0 
        
    if detour_A <= threshold and detour_B <= threshold:
        return True
    return False


# ── Core structures ───────────────────────────────────────────────
active_heap    = []                
active_rides   = {}                
removed_rids   = set()             
hex_to_rids    = defaultdict(set)  
batched_rids   = set()
seq = 0

max_active       = 0
total_evicted    = 0
snapshot_interval = max(1, len(df) // 1000)
snapshots        = []

best_sim_scores = [np.nan] * len(df)
best_sim_rids   = [None] * len(df)
batchable       = [0] * len(df)

columns = df.columns.tolist()

for i, (idx, row) in enumerate(df.iterrows()):
    current_ts    = row['ts']
    rid           = row['rid']
    curr_hex      = row['curr_hex']
    current_ts_ns = current_ts.value

    meta = ride_meta_sim.get(rid)
    if meta is None or not isinstance(meta.get('journey_hexes'), list) or not isinstance(meta.get('journey_coords'), list):
        continue

    end_time      = meta['trip_end_time']
    end_time_ns   = end_time.value

    while active_heap and active_heap[0][0] <= current_ts_ns:
        _, _, evict_rid = heapq.heappop(active_heap)
        if evict_rid in removed_rids:
            removed_rids.discard(evict_rid)
            continue
        if evict_rid in active_rides:
            evict_hex = active_rides[evict_rid].get('curr_hex')
            if evict_hex and evict_rid in hex_to_rids[evict_hex]:
                hex_to_rids[evict_hex].discard(evict_rid)
                if not hex_to_rids[evict_hex]:
                    del hex_to_rids[evict_hex]
            del active_rides[evict_rid]
            total_evicted += 1

    is_new_ride = rid not in active_rides

    if not is_new_ride:
        old_hex = active_rides[rid].get('curr_hex')
        if old_hex != curr_hex:
            if old_hex and rid in hex_to_rids[old_hex]:
                hex_to_rids[old_hex].discard(rid)
                if not hex_to_rids[old_hex]:
                    del hex_to_rids[old_hex]
            hex_to_rids[curr_hex].add(rid)
        removed_rids.add(rid)
    else:
        disk_hexes = h3.grid_disk(curr_hex, 1)
        neighbor_rids = []
        for h in disk_hexes:
            if h in hex_to_rids:
                neighbor_rids.extend(hex_to_rids[h])

        B_hexes = meta['journey_hexes']
        B_coords = meta['journey_coords']
        
        try:
            b_start_idx = B_hexes.index(curr_hex)
        except ValueError:
            b_start_idx = 0
            
        rem_B_hexes = B_hexes[b_start_idx:]
        rem_B_coords = [(row['lat'], row['lon']), B_coords[-1]] if len(B_coords) > 0 else []

        best_score = 0.0
        best_rid = None

        if len(neighbor_rids) > 0 and len(rem_B_hexes) > 0 and len(rem_B_coords) >= 2:
            for n_rid in neighbor_rids:
                if n_rid in batched_rids:
                    continue
                ar = active_rides[n_rid]
                
                A_hexes = ar['journey_hexes']
                A_coords = ar['journey_coords']
                A_curr_hex = ar['curr_hex']
                
                try:
                    a_start_idx = A_hexes.index(A_curr_hex)
                except ValueError:
                    a_start_idx = 0
                    
                rem_A_hexes = A_hexes[a_start_idx:]
                rem_A_coords = [(ar['lat'], ar['lon']), A_coords[-1]] if len(A_coords) > 0 else []

                if len(rem_A_hexes) > 0 and len(rem_A_coords) >= 2:
                    sim = path_similarity(rem_A_hexes, rem_B_hexes, rem_A_coords, rem_B_coords, k=2)
                    if sim > best_score:
                        best_score = sim
                        best_rid = n_rid
                        
        best_sim_scores[i] = best_score
        
        if best_score > 0.0 and best_rid is not None:
            best_sim_rids[i] = best_rid
            
            ar = active_rides[best_rid]
            A_curr = (ar['lat'], ar['lon'])
            B_curr = (row['lat'], row['lon'])
            
            A_end = (ar['trip_end_lat'], ar['trip_end_lon'])
            B_end = (meta['trip_end_lat'], meta['trip_end_lon'])
            
            is_batchable = check_osrm_detour(A_curr, B_curr, A_end, B_end, threshold=30.0)
            if is_batchable:
                batchable[i] = 1
                batched_rids.add(best_rid)
                batched_rids.add(rid)

        hex_to_rids[curr_hex].add(rid)

    ride_data = {c: row[c] for c in columns}
    ride_data['journey_hexes'] = meta['journey_hexes']
    ride_data['journey_coords'] = meta['journey_coords']
    ride_data['trip_end_lat'] = meta['trip_end_lat']
    ride_data['trip_end_lon'] = meta['trip_end_lon']

    active_rides[rid] = ride_data
    seq += 1
    heapq.heappush(active_heap, (end_time_ns, seq, rid))

    n_active = len(active_rides)
    if n_active > max_active:
        max_active = n_active

df['best_sim_score'] = best_sim_scores
df['best_sim_rid'] = best_sim_rids
df['batchable'] = batchable

print(f"✓ Done")
print(f"  Peak active rides         : {max_active}")
print(f"  Total evicted             : {total_evicted}")
print(f"  Still active (end)        : {len(active_rides)}")

batchable_df = df[df['batchable'] == 1]
print(f"  Batchable matches found   : {len(batchable_df)}")

print(len(batched_rids))
if len(batchable_df) > 0:
    print("\nSample batchable matches:")
    res = batchable_df[['ts', 'rid', 'curr_hex', 'best_sim_score', 'best_sim_rid', 'batchable']].head(10)
    print(res.to_string(index=False))


✓ Done
  Peak active rides         : 2747
  Total evicted             : 67945
  Still active (end)        : 82
  Batchable matches found   : 11539
23062

Sample batchable matches:
                     ts                                  rid        curr_hex  best_sim_score                         best_sim_rid  batchable
2025-08-02 00:21:22.436 8085f759-a682-457c-97f7-5a0bce92cf00 896014594a7ffff        0.222222 efc8aefb-04d6-41cc-af89-09a8bff1e2dd          1
2025-08-02 00:22:59.878 ffa14a9a-7269-4e93-8cb4-71df773ae08d 8960145b117ffff        0.545455 6ee06f44-0858-4e24-9cfe-07dde3997e50          1
2025-08-02 00:30:02.969 ab53363b-eeac-4da9-9f29-dbd36e1fcd86 896189272d3ffff        0.595745 ddb5e5ee-b4a9-4eea-accd-53dc8d7e1aee          1
2025-08-02 00:32:10.725 aaaab6d4-6567-4dee-a0ca-534ed7c6f58c 8961892ea17ffff        0.857143 050e04e0-73c2-403d-bb92-a20975b86838          1
2025-08-02 00:35:34.726 b9d2da49-ea10-465b-8ccc-ec6d1c0c0dfb 8960145b57bffff        0.971429 8392c456-912e-42f4-bd6